In [1]:
# Backend
import os

os.environ["KERAS_BACKEND"] = "jax"

In [2]:
# Setup
import json
from importlib import import_module
from pathlib import Path

import numpy as np
from rich.table import Table

from src.callbacks import ImportanceUpdate
from src.models import GradientBoostedDecisionTree as BDT
from src.models import LearnableCutFlowParallel as LCF_PAR
from src.models import LearnableCutFlowSequential as LCF_SEQ
from src.models import MultiLayerPerceptron as MLP
from src.utils import Timer, load_model, print

In [3]:
# Parameters: entered
rerun = False
n_runs = 10

# Dataset
feature_selection = [0, 1, 2, 3, 4, 5]  # [0, 1, 2, 3, 4, 5], [0, 3, 4], [0, 2, 4, 5]

# Path
FIGURES_DIR = Path("figures")
CHECKPOINTS_DIR = Path("checkpoints")
RESULTS_DIR = Path("results")

In [4]:
# Parameters: derived
# Config
with open("1-dataset:config.json", "r") as f:
    dataset_config = json.load(f)

with open("2-model:config.json", "r") as f:
    model_config = json.load(f)

# Dataset
dataset_name = "real1"
features = dataset_config[dataset_name]["features"]
n_samples = dataset_config[dataset_name]["n_samples"]
seed = dataset_config[dataset_name]["seed"]

# Model
centers = [80, 0.15, 0.025, 2, 2, 0.3]
n_epochs = 200
batch_size = 512

# Figure
bins = [np.linspace(**kwargs) for kwargs in dataset_config[dataset_name]["bins"]]

# Path
prefix = f"4-dataset:{dataset_name}-model:bdt,mlp,lcf-selection:{''.join([str(i) for i in feature_selection])}-"
suffix = f"(x{n_runs})"

# Results
results_path = RESULTS_DIR / f"{prefix}results{suffix}.json"

is_complete = results_path.exists()
rerun = rerun or not is_complete

if rerun:
    results = {}
else:
    with open(results_path, "r") as f:
        results = json.load(f)

In [5]:
# Dataset
module = import_module(f"src.datasets.{dataset_name}")
load_data = getattr(module, "load_data")
(x_train, y_train), (x_test, y_test) = load_data(n_samples, seed)

selection = np.ones_like(x_train[:, 0], dtype=bool)
for i in range(x_train.shape[1]):
    p05 = np.percentile(x_train[:, i], 5)
    p95 = np.percentile(x_train[:, i], 95)
    selection = (p05 < x_train[:, i]) & (x_train[:, i] < p95) & selection
x_train = x_train[selection]
y_train = y_train[selection]

selection = np.ones_like(x_test[:, 0], dtype=bool)
for i in range(x_test.shape[1]):
    p05 = np.percentile(x_test[:, i], 5)
    p95 = np.percentile(x_test[:, i], 95)
    selection = (p05 < x_test[:, i]) & (x_test[:, i] < p95) & selection
x_test = x_test[selection]
y_test = y_test[selection]

print(f"{x_train.shape=}")
print(f"{y_train.shape=}")
print(f"{x_test.shape=}")
print(f"{y_test.shape=}")

x_train.shape=(75549, 6)
y_train.shape=(75549, 1)
x_test.shape=(75436, 6)
y_test.shape=(75436, 1)


In [6]:
# Model: BDT
if rerun:
    results["bdt"] = {
        "file_path": {"records": []},
        "training_time": {"records": [], "mean": None, "std": None},
    }

    for i in range(n_runs):
        print(f"Processing bdt@{i + 1}...")

        bdt = BDT(input_shape=x_train.shape, name=f"bdt@{i + 1}")
        bdt.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            bdt.fit(
                x_train,
                y_train.squeeze(),
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=0,
            )

        file_path = CHECKPOINTS_DIR / f"{prefix}bdt@{i + 1}{suffix}.pkl"
        bdt.save(file_path)

        results["bdt"]["file_path"]["records"].append(file_path.as_posix())
        results["bdt"]["training_time"]["records"].append(timer.record)

    results["bdt"]["training_time"]["mean"] = np.mean(
        results["bdt"]["training_time"]["records"]
    )
    results["bdt"]["training_time"]["std"] = np.std(
        results["bdt"]["training_time"]["records"]
    )

training_time_mean = results["bdt"]["training_time"]["mean"]
training_time_std = results["bdt"]["training_time"]["std"]
print(f"{training_time_mean:.2f} ± {training_time_std:.2f} seconds")

18.32 ± 0.06 seconds


In [7]:
# Model: MLP
if rerun:
    results["mlp"] = {
        "file_path": {"records": []},
        "training_time": {"records": [], "mean": None, "std": None},
    }

    for i in range(n_runs):
        print(f"Processing mlp@{i + 1}...")

        mlp = MLP(x_train.shape, name=f"mlp@{i + 1}")
        mlp.adapt(x_train)
        mlp.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            mlp.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=0,
            )

        file_path = CHECKPOINTS_DIR / f"{prefix}mlp@{i + 1}{suffix}.keras"
        mlp.save(file_path)

        results["mlp"]["file_path"]["records"].append(file_path.as_posix())
        results["mlp"]["training_time"]["records"].append(timer.record)

    results["mlp"]["training_time"]["mean"] = np.mean(
        results["mlp"]["training_time"]["records"]
    )
    results["mlp"]["training_time"]["std"] = np.std(
        results["mlp"]["training_time"]["records"]
    )

training_time_mean = results["mlp"]["training_time"]["mean"]
training_time_std = results["mlp"]["training_time"]["std"]
print(f"{training_time_mean:.2f} ± {training_time_std:.2f} seconds")

33.48 ± 1.56 seconds


In [8]:
# Model: LCF(parallel)
if rerun:
    results["lcf_par"] = {
        "file_path": {"records": []},
        "training_time": {"records": [], "mean": None, "std": None},
    }

    for i in range(n_runs):
        print(f"Processing lcf_par@{i + 1}...")

        lcf_par = LCF_PAR(
            x_train.shape,
            centers,
            features=features,
            name=f"lcf_par@{i + 1}",
        )
        lcf_par.adapt(x_train)
        lcf_par.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            lcf_par.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                callbacks=[ImportanceUpdate(lcf_par)],
                verbose=0,
            )

        file_path = CHECKPOINTS_DIR / f"{prefix}lcf_par@{i + 1}{suffix}.keras"
        lcf_par.save(file_path)

        results["lcf_par"]["file_path"]["records"].append(file_path.as_posix())
        results["lcf_par"]["training_time"]["records"].append(timer.record)

    results["lcf_par"]["training_time"]["mean"] = np.mean(
        results["lcf_par"]["training_time"]["records"]
    )
    results["lcf_par"]["training_time"]["std"] = np.std(
        results["lcf_par"]["training_time"]["records"]
    )

training_time_mean = results["lcf_par"]["training_time"]["mean"]
training_time_std = results["lcf_par"]["training_time"]["std"]
print(f"{training_time_mean:.2f} ± {training_time_std:.2f} seconds")

39.51 ± 0.78 seconds


In [9]:
# Model: LCF(sequential)
if rerun:
    results["lcf_seq"] = {
        "file_path": {"records": []},
        "training_time": {"records": [], "mean": None, "std": None},
    }

    for i in range(n_runs):
        print(f"Processing lcf_seq@{i + 1}...")

        lcf_seq = LCF_SEQ(
            x_train.shape,
            centers,
            features=features,
            name=f"lcf_seq@{i + 1}",
        )
        lcf_seq.adapt(x_train)
        lcf_seq.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            lcf_seq.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                callbacks=[ImportanceUpdate(lcf_seq)],
                verbose=0,
            )

        file_path = CHECKPOINTS_DIR / f"{prefix}lcf_seq@{i + 1}{suffix}.keras"
        lcf_seq.save(file_path)

        results["lcf_seq"]["file_path"]["records"].append(file_path.as_posix())
        results["lcf_seq"]["training_time"]["records"].append(timer.record)

    results["lcf_seq"]["training_time"]["mean"] = np.mean(
        results["lcf_seq"]["training_time"]["records"]
    )
    results["lcf_seq"]["training_time"]["std"] = np.std(
        results["lcf_seq"]["training_time"]["records"]
    )

training_time_mean = results["lcf_seq"]["training_time"]["mean"]
training_time_std = results["lcf_seq"]["training_time"]["std"]
print(f"{training_time_mean:.2f} ± {training_time_std:.2f} seconds")

40.89 ± 1.23 seconds


In [10]:
# Checkpoints
# fmt: off
bdt_ckpts = [load_model(Path(ckpt)) for ckpt in results["bdt"]["file_path"]["records"]]
mlp_ckpts = [load_model(Path(ckpt)) for ckpt in results["mlp"]["file_path"]["records"]]
lcf_par_ckpts = [load_model(Path(ckpt)) for ckpt in results["lcf_par"]["file_path"]["records"]]
lcf_seq_ckpts = [load_model(Path(ckpt)) for ckpt in results["lcf_seq"]["file_path"]["records"]]
# fmt: on

checkpoints = [bdt_ckpts, mlp_ckpts, lcf_par_ckpts, lcf_seq_ckpts]

In [11]:
# Analysis: metrics
y_true = y_test

table = Table(title="Model Performance Comparison")
table.add_column("#", justify="center", style="cyan", no_wrap=True)
table.add_column("Model", style="magenta")
table.add_column("TP", justify="right", style="green")
table.add_column("FP", justify="right", style="red")
table.add_column("Accuracy", justify="right", style="blue")
table.add_column("Precision", justify="right", style="blue")
table.add_column("Significance", justify="right", style="yellow")
table.add_column("Time(s)", justify="right", style="yellow")

if rerun:
    results["bdt"].update(
        {
            "tp": {"records": [], "mean": None, "std": None},
            "fp": {"records": [], "mean": None, "std": None},
            "accuracy": {"records": [], "mean": None, "std": None},
            "precision": {"records": [], "mean": None, "std": None},
            "significance": {"records": [], "mean": None, "std": None},
        }
    )
    results["mlp"].update(
        {
            "tp": {"records": [], "mean": None, "std": None},
            "fp": {"records": [], "mean": None, "std": None},
            "accuracy": {"records": [], "mean": None, "std": None},
            "precision": {"records": [], "mean": None, "std": None},
            "significance": {"records": [], "mean": None, "std": None},
        }
    )
    results["lcf_par"].update(
        {
            "tp": {"records": [], "mean": None, "std": None},
            "fp": {"records": [], "mean": None, "std": None},
            "accuracy": {"records": [], "mean": None, "std": None},
            "precision": {"records": [], "mean": None, "std": None},
            "significance": {"records": [], "mean": None, "std": None},
        }
    )
    results["lcf_seq"].update(
        {
            "tp": {"records": [], "mean": None, "std": None},
            "fp": {"records": [], "mean": None, "std": None},
            "accuracy": {"records": [], "mean": None, "std": None},
            "precision": {"records": [], "mean": None, "std": None},
            "significance": {"records": [], "mean": None, "std": None},
        }
    )

    for i, model_name in enumerate(results):
        print(f"Processing {model_name}...")

        for checkpoint in checkpoints[i]:
            y_pred = checkpoint.predict(x_test, batch_size=batch_size, verbose=0)
            y_pred = np.all(y_pred > 0.5, axis=1, keepdims=True)

            tp = np.sum((y_true == 1) & (y_pred == 1))
            fp = np.sum((y_true == 0) & (y_pred == 1))
            tn = np.sum((y_true == 0) & (y_pred == 0))
            fn = np.sum((y_true == 1) & (y_pred == 0))

            accuracy = (tp + tn) / (tp + tn + fp + fn)
            precision = tp / (tp + fp)

            s = tp / (tp + tn + fp + fn) * 3000 * 1000 * 0.7644
            b = fp / (tp + tn + fp + fn) * 3000 * 1000 * 1.806 * 1e5
            significance = s / np.sqrt(b)

            results[model_name]["tp"]["records"].append(tp.tolist())
            results[model_name]["fp"]["records"].append(fp.tolist())
            results[model_name]["accuracy"]["records"].append(accuracy.tolist())
            results[model_name]["precision"]["records"].append(precision.tolist())
            results[model_name]["significance"]["records"].append(significance.tolist())

        # fmt: off
        results[model_name]["tp"]["mean"] = np.mean(results[model_name]["tp"]["records"])
        results[model_name]["tp"]["std"] = np.std(results[model_name]["tp"]["records"])
        results[model_name]["fp"]["mean"] = np.mean(results[model_name]["fp"]["records"])
        results[model_name]["fp"]["std"] = np.std(results[model_name]["fp"]["records"])
        results[model_name]["accuracy"]["mean"] = np.mean(results[model_name]["accuracy"]["records"])
        results[model_name]["accuracy"]["std"] = np.std(results[model_name]["accuracy"]["records"])
        results[model_name]["precision"]["mean"] = np.mean(results[model_name]["precision"]["records"])
        results[model_name]["precision"]["std"] = np.std(results[model_name]["precision"]["records"])
        results[model_name]["significance"]["mean"] = np.mean(results[model_name]["significance"]["records"])
        results[model_name]["significance"]["std"] = np.std(results[model_name]["significance"]["records"])
        # fmt: on

for i, (name, metrics) in enumerate(results.items()):
    tp_mean = metrics["tp"]["mean"]
    tp_std = metrics["tp"]["std"]
    fp_mean = metrics["fp"]["mean"]
    fp_std = metrics["fp"]["std"]
    accuracy_mean = metrics["accuracy"]["mean"]
    accuracy_std = metrics["accuracy"]["std"]
    precision_mean = metrics["precision"]["mean"]
    precision_std = metrics["precision"]["std"]
    significance_mean = metrics["significance"]["mean"]
    significance_std = metrics["significance"]["std"]
    training_time_mean = metrics["training_time"]["mean"]
    training_time_std = metrics["training_time"]["std"]

    table.add_row(
        str(i + 1),
        name,
        f"{tp_mean:.0f}\n± {tp_std:.0f}",
        f"{fp_mean:.0f}\n± {fp_std:.0f}",
        f"{accuracy_mean:.4f}\n± {accuracy_std:.4f}",
        f"{precision_mean:.4f}\n± {precision_std:.4f}",
        f"{significance_mean:.4f}\n± {significance_std:.4f}",
        f"{training_time_mean:.2f}\n± {training_time_std:.2f}",
    )

print(table)

                         Model Performance Comparison                          
┏━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ # ┃ Model   ┃    TP ┃    FP ┃ Accuracy ┃ Precision ┃ Significance ┃ Time(s) ┃
┡━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ 1 │ bdt     │ 41172 │  6143 │   0.8753 │    0.8702 │       5.9586 │   18.32 │
│   │         │   ± 0 │   ± 0 │ ± 0.0000 │  ± 0.0000 │     ± 0.0002 │  ± 0.06 │
│ 2 │ mlp     │ 40819 │  5480 │   0.8794 │    0.8817 │       6.2595 │   33.48 │
│   │         │ ± 372 │ ± 285 │ ± 0.0013 │  ± 0.0045 │     ± 0.1077 │  ± 1.56 │
│ 3 │ lcf_par │ 24392 │  2838 │   0.6966 │    0.8958 │       5.1933 │   39.51 │
│   │         │  ± 62 │  ± 12 │ ± 0.0007 │  ± 0.0002 │     ± 0.0074 │  ± 0.78 │
│ 4 │ lcf_seq │ 40491 │  6619 │   0.8599 │    0.8595 │       5.6454 │   40.89 │
│   │         │  ± 29 │  ± 31 │ ± 0.0001 │  ± 0.0005 │     ± 0.0095 │  ± 1.23 │
└───┴─────────┴───────┴───────┴─────────

In [12]:
# Results
if rerun:
    with open(RESULTS_DIR / f"{prefix}results{suffix}.json", "w") as f:
        json.dump(results, f, indent=4)